<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/11_1_Single_and_Multi_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[[실습 11-1] 단일 에이전트 vs 멀티 에이전트 시스템 구현  

### 실습목표

- 단일 에이전트의 GRAM(Goal-Reasoning-Action-Memory) 루프를 코드로 구현하고 작동 원리를 이해한다 [cite: 11-1-1].  

- 단일 에이전트가 복잡한 작업 수행 시 겪는 '추론 부하' 문제를 확인한다 [cite: 11-1-1].  

- 역할을 분리한 멀티 에이전트(Multi-Agent) 구조와 이를 관리하는 Orchestration(조율) 로직을 구현하여 성능 향상을 체험한다 [cite: 11-1-2, 11-1-3].  

1. 환경 준비 및 라이브러리 설치  

- 실습을 위해 LangChain 및 Google Gemini 설정을 진행합니다.  

In [ ]:
# 실습을 위한 라이브러리 설치
!pip install -q -U langchain langchain-google-genai langchain-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.5/236.5 kB 14.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.48.0 which is incompatible.


In [ ]:
# Key 생성 및 설정 방법 참고: https://wikidocs.net/328722
# Google API Key 설정
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API 설정 완료")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini API 설정 완료


In [ ]:
# 1. 필수 라이브러리 설치
# !pip install -q -U langchain langchain-community langchain-google-genai

import google.generativeai as genai
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Gemini API 설정
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', api_key=GOOGLE_API_KEY)

2. [파트 1] 단일 에이전트 구현 (GRAM 루프)

- 하나의 LLM이 모든 역할을 수행하는 단일 에이전트 구조입니다. 복잡한 요청(기획 + 초안 작성)을 한꺼번에 처리할 때의 결과를 관찰합니다 [cite: 11-1-1].  

In [ ]:
# --- 단일 에이전트 프롬프트 (Goal + Reasoning 요구) --- [cite: 11-1-1]
single_agent_prompt = ChatPromptTemplate.from_template("""
너는 전천후 AI 비서야. 아래 요청에 대해 [기획]과 [작성]을 모두 혼자서 수행해줘.

요청: {request}

형식:
1. 기획: 작업 단계별 계획
2. 작성: 실제 결과물 내용
""")

single_agent_chain = single_agent_prompt | llm | StrOutputParser()

# 실행 테스트
request_text = "AI 에이전트의 미래에 대한 기술 블로그 포스트를 기획하고 작성해줘."
print("--- [단일 에이전트 실행 결과] ---")
print(single_agent_chain.invoke({"request": request_text}))

--- [단일 에이전트 실행 결과] ---
## AI 에이전트의 미래에 대한 기술 블로그 포스트 기획 및 작성

---

## 1. 기획: 작업 단계별 계획

| 단계 | 세부 내용 | 목표 |
|---|---|---|
| **1단계: 주제 및 목표 설정** | 주제: AI 에이전트의 미래 (자율성, 멀티모달리티, 실제 적용) | 독자에게 AI 에이전트 기술의 현재와 미래 비전을 명확하게 전달하고 흥미를 유발 |
| **2단계: 독자 분석 및 톤 설정** | 독자: 기술에 관심 있는 일반인, 개발자, IT 종사자. 톤: 전문적이지만 이해하기 쉬운, 미래 지향적이고 긍정적인 톤 | 기술적 깊이와 대중적 접근성 균형 유지 |
| **3단계: 목차 구성** | 도입 (현황 및 정의), 본론 1 (주요 미래 기술), 본론 2 (실제 적용 사례), 결론 (미래 전망 및 시사점) | 논리적이고 일관성 있는 글의 흐름 확보 |
| **4단계: 핵심 내용 개발** | **본론 1 (미래 기술):** 자율 학습/판단, 멀티 에이전트 시스템, 멀티모달 상호작용 강조 | 기술적 혁신 요소를 구체적으로 제시 |
| | **본론 2 (적용 사례):** 개인화된 비서, 산업 자동화, 복잡한 문제 해결 (R&D) | 기술이 실생활과 산업에 미칠 영향을 시각화 |
| **5단계: 제목 및 인트로/결론 작성** | 흥미로운 제목 선정. 인트로에서 독자의 호기심 자극. 결론에서 강력한 메시지 전달 | 글의 몰입도 및 완결성 높이기 |
| **6단계: 검토 및 수정** | 전문 용어의 적절성, 문장 흐름, 전체적인 메시지 전달력 확인 | 최종 결과물의 품질 확보 |

---

## 2. 작성: 실제 결과물 내용

### [기술 블로그 포스트]

# 🤖 자율성의 시대로: AI 에이전트의 미래, 어디까지 왔고 어디로 가는가

## 🚀 서론: 단순한 챗봇을 넘어, '자율적 행위자'의 등장

우리는 이미 'AI 비서'와 익숙합니다. 스마트폰의 음성 인식 기능부터 고객 서비스 챗봇까지, AI는 우리의 

- **관찰 포인트**: 단일 에이전트는 기획과 작성을 한 번에 수행하므로, 기획 내용이 부실하거나 작성 내용이 기획과 어긋나는 '추론 혼선'이 발생할 가능성이 높습니다 [cite: 11-1-1].  

3. [파트 2] 멀티 에이전트 및 Orchestration 구현  

- 역할을 분리하고 상위에서 흐름을 제어하는 Orchestration 구조를 구현합니다 [cite: 11-1-3].  

    (1) 역할별 에이전트 정의 [cite: 11-1-2]  

In [ ]:
# 에이전트 1: 기획 전문가 (Planner)
planner_prompt = ChatPromptTemplate.from_template("너는 콘텐츠 기획 전문가야. '{topic}'에 대한 블로그 포스트의 목차와 핵심 키워드를 기획해줘.")
planner_agent = planner_prompt | llm | StrOutputParser()

# 에이전트 2: 작가 (Writer)
writer_prompt = ChatPromptTemplate.from_template("너는 전문 작가야. 다음 기획안을 바탕으로 블로그 본문을 작성해줘.\n\n[기획안]\n{plan}")
writer_agent = writer_prompt | llm | StrOutputParser()

(2) Orchestration (조율) 로직 구현 [cite: 11-1-3]  

- 에이전트 간의 메시지 흐름과 실행 순서를 관리하는 Controller 역할을 함수로 정의합니다.  

In [ ]:
def orchestrator(topic):
    print(f"[*] 단계 1: 기획 에이전트 실행 중... (Goal: {topic})")
    # 1. 기획 에이전트 호출 [cite: 11-1-3]
    plan = planner_agent.invoke({"topic": topic})
    print(f"[*] 기획 완료 (Memory 저장)")

    print(f"[*] 단계 2: 작성 에이전트 실행 중... (Communication)")
    # 2. 기획 결과를 작성 에이전트에게 전달 [cite: 11-1-3]
    final_post = writer_agent.invoke({"plan": plan})

    return {
        "plan": plan,
        "final_post": final_post
    }

# 실행 테스트
print("\n--- [멀티 에이전트 시스템 실행] ---")
result = orchestrator("AI 에이전트의 미래")
print(f"\n[최종 결과물]:\n{result['final_post']}")


--- [멀티 에이전트 시스템 실행] ---
[*] 단계 1: 기획 에이전트 실행 중... (Goal: AI 에이전트의 미래)
[*] 기획 완료 (Memory 저장)
[*] 단계 2: 작성 에이전트 실행 중... (Communication)

[최종 결과물]:
# 단순한 챗봇을 넘어: 자율 경제를 이끌 AI 에이전트의 미래와 윤리적 딜레마

---
**[메인 키워드: AI 에이전트, 자율형 AI, 멀티 에이전트 시스템]**

## 💡 프롤로그: 단순한 챗봇을 넘어, '자율적 행동'을 시작한 AI

지난 몇 년간 우리는 대규모 언어 모델(LLM)의 등장으로 인공지능이 얼마나 강력한 '지능'을 가질 수 있는지 목격했습니다. 그러나 이 지능은 대부분 수동적이었습니다. 사용자가 질문을 던지면 답변하는, 이른바 '요청-응답' 모델이었죠.

이제 패러다임이 바뀌고 있습니다.

AI는 단순한 대화 상대나 정보 검색 도구를 넘어, 스스로 목표를 설정하고, 복잡한 계획을 수립하며, 필요한 도구를 사용해 목표를 달성하는 ‘**자율적 행동(Autonomous Action)**’을 시작했습니다. 이것이 바로 우리가 주목해야 할 미래 기술, **AI 에이전트(AI Agent)**입니다.

AI 에이전트는 디지털 환경에서 인간의 개입을 최소화하며 업무를 수행하는 차세대 인공지능입니다. 이 글은 AI 에이전트의 작동 원리를 심층 분석하고, 이 기술이 우리의 노동, 경제, 그리고 윤리적 영역에 가져올 혁신과 도전 과제를 통찰력 있게 제시합니다.

---

## 1. AI 에이전트란 무엇인가? 개념과 작동 원리

### 1-1. 에이전트의 정의: LLM과의 결정적 차이

많은 독자들이 AI 에이전트를 챗봇의 진화형으로 생각하지만, 둘 사이에는 근본적인 차이가 있습니다.

*   **LLM (Large Language Model):** AI 에이전트의 **'뇌(Brain)'** 역할을 합니다. 방대한 데이터를 학습하여 언어를 이해하고, 추론하며, 명령을 처리하는 지능의 핵심입니다.
*   

4. 실습 결과 비교 및 분석  

| 구분 | 단일 에이전트 (Single) | 멀티 에이전트 (MAS) |
|------|----------------------|----------------------|
| 추론 집중도 | 기획과 작성을 동시에 고민함 | 기획과 작성을 단계별로 나누어 집중함 |
| 결과물 품질 | 전반적으로 평이하거나 모호함 | 기획에 기반한 구체적인 서술이 가능함 |
| 구조적 특징 | GRAM 루프가 1회 발생 | 에이전트 간 결과 전달(Communication) 발생 |
